In [ ]:
-- Ver só primeiras n linhas:
    SELECT * FROM df1 limit 10;

-- Empilhar bases:
    SELECT * FROM df1 UNION ALL df2;

-- Selecionar a linha da base em que o CPF tem máxima renda:

    -- Abordagem1: cria uma base com a renda máxima do CPF >> verifica qual é o primeiro mês que aparece essa renda máxima >> traz todas as linhas deste mês
    select * from diamond.ds_ren_pf_02_score as a1 right join
        -- base com o primeiro mês que aparece a renda máxima
        (select a.num_cpf, min(a.dat_referencia) as primeiro_mes_max_renda
            from diamond.ds_ren_pf_02_score as a right join
                -- base com a renda máxima
                (select num_cpf, max(vlr_renda_risco) as vlr_max_renda from diamond.ds_ren_pf_02_score where num_cpf in (39521708875,35510892803) group by 1) as max_renda
            on a.num_cpf = max_renda.num_cpf and a.vlr_renda_risco = vlr_max_renda
        group by 1) as df_prim_mes_max_renda
    on a1.num_cpf = df_prim_mes_max_renda.num_cpf and a1.dat_referencia = df_prim_mes_max_renda.primeiro_mes_max_renda

    -- Abordagem2: particiona por CPF, renda e mês, ordenando renda do maior para o menor e mês do menor para o maior, marcando a ordem >> seleciona o primeiro rank, que já será por CPF a maior renda no primeiro mês que aparece
    select * from
        (select *,
                ROW_NUMBER() OVER (PARTITION BY num_cpf ORDER BY vlr_renda_risco DESC, dat_referencia ASC) as rank
        from diamond.ds_ren_pf_02_score where num_cpf in (39521708875,35510892803))
    where rank=1

--1. Aggregation clauses
--      COUNT, SUM, AVG, MIN, MAX, COUNT(DISTINCT var1)
--      GROUP BY (var1)
--      HAVING (count(*) > n)  -- Filtra após o agrupamento. Se aplicar where, filtra antes do agrupamento

--2. Window functions (over partition by)
--      ROW_NUMBER() OVER (partition by var1) as rownumber (atribui um número de linha a cada registro dentro de uma partição)
--      RANK() OVER (partition by var1) as rank (atribui um número de linha a cada registro dentro de uma partição, mas registra o mesmo ranking para empates)
--      DENSE_RANK() OVER (partition by var1) as dense_rank (similar ao RANK, mas não pula números em caso de empates)
--      SUM(var2) OVER (partition by var1) as sum_var2 (calcula a soma dentro de uma partição sem agrupar os resultados em uma única linha)
--      AVG(var2) OVER (partition by var1) as avg_var2 (calcula a média dentro de uma partição sem agrupar os resultados em uma única linha)

--3. Types of join (left, right, cross, inner) and how to use them
--      INNER JOIN
--      LEFT JOIN
--      RIGHT JOIN
--      FULL JOIN
--      CROSS JOIN (produto cartesiano)
--      SELF JOIN (join da tabela com ela mesma)

--      Append: SELECT * FROM a UNION ALL SELECT * FROM b

--4. Case clause with aggregate functions
--      CASE WHEN condição THEN resultado end as NEW_VAR
--      sum(CASE WHEN condição THEN resultado end) as SUM_NEW_VAR

--5. Time functions (e.g., datediff, datetrunc)
--      FORMAT: formatar
                    -- SELECT FORMAT('2024-10-15', 'dd/MM/yyyy') AS formatted_date
--      DATEDIFF: qtd. de períodos no intervalo
                    -- SELECT DATEDIFF(day, '2024-01-01', '2024-10-15') AS days_difference
--      DATE_TRUNC: início do mês
                    -- SELECT DATE_TRUNC('month', '2024-10-15') AS month_start (início do mês)
--      GETDATE(): puxar atual
                    -- SELECT GETDATE() AS current_datetime
--      DATEADD: adicionar período
                    -- SELECT DATEADD(day, 10, '2024-10-15') AS new_date
--      YEAR, MONTH, DAY: extrair info
                    -- SELECT YEAR('2024-10-15') AS year
--      DATE_PART: extrair info, mais opções
                    -- SELECT DATE_PART('quarter', '2024-10-15') AS quarter;
--      EXTRACT: extrair info de timestamp
                    -- SELECT EXTRACT(HOUR FROM '2024-10-15 15:30:00') AS hour;
--                  -- SELECT EXTRACT(EPOCH FROM (transaction_timestamp - time_ant)) as dif (em segundos)

--6. Outros:
--      SELECT CAST('2024-10-15 15:30:00' AS DATE) AS date_only;
--      SELECT CAST('123' AS INT) AS numeric_value;
--      SELECT CAST(123 AS VARCHAR) AS string_value;
--      SELECT SUBSTRING('Hello World', 1, 5) AS substring_result;
--      SELECT REPLACE('Hello World', 'H', 'X') -- 'Xello World'
--      SELECT LPAD(CAST(123 AS VARCHAR), 6, '0') AS padded_string; adiciona 0 a esqueda até preencher 6 números
--      FLOOR(var): arrendonda pra baixo
--      CEIL(var): arrendonda pra cima
--      A%B: resto da divisão
--      concat(topping_name1,',',topping_name2,',',topping_name3) as pizza
--      LAG(expression, [offset], [default]) OVER (PARTITION BY partition_column ORDER BY order_column)
            --offset = default é 1, quantas linhas quer pular
            --default = qual valor colocar quando não acha valor
            --partition by é opcional
            --order by é order para rodar a função

## CTE (common table expressions)
cria tabelas temporárias para serem usadas - precisar rodar junto com o código

In [ ]:
%sql
-- CTE + union all
WITH exampleCTE AS (
    SELECT *
    FROM diamond.ds_score_cpf
    WHERE num_cpf in ('39521708875','35510892803')
)
SELECT num_cpf, dat_processamento, vlr_score_cpf
from exampleCTE
where num_cpf = '39521708875' and vlr_score_cpf > 700

UNION ALL

SELECT num_cpf, dat_processamento, vlr_score_cpf
from exampleCTE
where num_cpf = '35510892803' and vlr_score_cpf > 600


-- CTE + over partition
WITH exampleCTE AS (
    SELECT *
    FROM diamond.ds_score_cpf
    WHERE num_cpf in ('39521708875','35510892803')
)

select num_cpf,
       dat_processamento,
       vlr_score_cpf,
       row_number() over (partition by num_cpf order by vlr_score_cpf desc) as row_number,
       rank() over (partition by num_cpf order by vlr_score_cpf desc) as rank,
       dense_rank() over (partition by num_cpf order by vlr_score_cpf desc) as denserank

 from exampleCTE


 -- CTEagg
WITH exampleCTEagg AS (
    SELECT num_cpf,
            date_trunc('month',dat_processamento) as mes,
            --month(dat_processamento) as mes,
            avg(vlr_score_cpf) as score_medio
        FROM diamond.ds_score_cpf
            WHERE num_cpf in ('39521708875','35510892803')
                GROUP BY num_cpf,mes
)
SELECT * FROM exampleCTEagg
order by num_cpf, mes


-- CTEagg + OVER PARTITION
WITH exampleCTEagg AS (
    SELECT num_cpf,
            month(dat_processamento) as mes,
            avg(vlr_score_cpf) as score_medio
        FROM diamond.ds_score_cpf
            WHERE num_cpf in ('39521708875','35510892803')
                GROUP BY num_cpf,mes
)
SELECT num_cpf,
       mes,
       sum(score_medio) over (partition by num_cpf) as soma_score_medio
 FROM exampleCTEagg
    order by num_cpf, mes


-- CTErecursive
WITH RECURSIVE EmployeeHierarchy AS (
    -- seleciona todo mundo que é gerente (náo tem gerente acima preenchido)
    SELECT employee_id, manager_id, first_name
    FROM employees
    WHERE manager_id IS NULL

    UNION ALL

    -- seleciona todo os empregados em que o gerente está na base acima
    SELECT e.employee_id, e.manager_id, e.first_name
    FROM employees e
    INNER JOIN EmployeeHierarchy eh ON e.manager_id = eh.employee_id
)

SELECT * FROM EmployeeHierarchy;